In [1]:
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [2]:
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()


def get_movie_details(movie_id):
    response = requests.get(f"{BASE_URL}/movies/{movie_id}")
    return response.json()


def get_similar_movies(movie_id):
    response = requests.get(f"{BASE_URL}/movies/{movie_id}/similar")
    return response.json()

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID, including overview, genres, budget, revenue, runtime, and rating.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie.",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get a list of movies similar to a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "integer",
                        "description": "The ID of the movie.",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
]

available_functions = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [4]:
system_prompt = """You are a movie expert assistant who speaks Korean.
Use the provided tools to look up movie information and answer user questions.
Remember the context of the conversation, including movie IDs mentioned in previous turns.
When the user refers to a movie discussed earlier, use the correct movie ID from the conversation history."""

messages = [{"role": "system", "content": system_prompt}]


def run_agent(user_message):
    print(f"User: {user_message}\n")
    messages.append({"role": "user", "content": user_message})

    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )

        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:
            print(f"Agent: {message.content}")
            return message.content

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            print(f"  -> Calling: {function_name}({function_args})")

            result = available_functions[function_name](**function_args)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )

In [5]:
run_agent("지금 인기 있는 영화 알려줘")

User: 지금 인기 있는 영화 알려줘



  -> Calling: get_popular_movies({})


Agent: 현재 인기 있는 영화 몇 편을 소개해 드릴게요.

1. **[Your Heart Will Be Broken](https://www.themoviedb.org/movie/1523145)**
   - 개봉일: 2026-03-26
   - 장르: 로맨스, 드라마
   - 줄거리: 고등학생 폴리나가 학교에서 괴롭힘을 당하다가 주범인 바르스와 사귀기로 하며 진짜 감정이 생기는 이야기입니다.
   - 평점: 6.9
   ![포스터](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)

2. **[Avatar: Fire and Ash](https://www.themoviedb.org/movie/83533)**
   - 개봉일: 2025-12-17
   - 장르: SF, 모험, 판타지
   - 줄거리: 제이크 설리와 네이티리의 가족이 새롭게 등장한 위협에 맞서 싸우는 이야기를 다룹니다.
   - 평점: 7.373
   ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **[Hoppers](https://www.themoviedb.org/movie/1327819)**
   - 개봉일: 2026-03-04
   - 장르: 애니메이션, 가족, SF, 코미디, 모험
   - 줄거리: 사람의 의식을 로봇 동물로 전이하는 과학 기술을 통해 동물과 소통하는 이야기입니다.
   - 평점: 7.6
   ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **[The Super Mario Bros. Movie](https://www.themoviedb.org/movie/502356)**
   - 개봉일: 2023-04-05
   - 장르: 가족, 코미디, 모험, 판타지, 애니메이션
   - 줄거리: 브루클린의 배관공 형제 마리오와 루이지가 신비한 세

'현재 인기 있는 영화 몇 편을 소개해 드릴게요.\n\n1. **[Your Heart Will Be Broken](https://www.themoviedb.org/movie/1523145)**\n   - 개봉일: 2026-03-26\n   - 장르: 로맨스, 드라마\n   - 줄거리: 고등학생 폴리나가 학교에서 괴롭힘을 당하다가 주범인 바르스와 사귀기로 하며 진짜 감정이 생기는 이야기입니다.\n   - 평점: 6.9\n   ![포스터](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)\n\n2. **[Avatar: Fire and Ash](https://www.themoviedb.org/movie/83533)**\n   - 개봉일: 2025-12-17\n   - 장르: SF, 모험, 판타지\n   - 줄거리: 제이크 설리와 네이티리의 가족이 새롭게 등장한 위협에 맞서 싸우는 이야기를 다룹니다.\n   - 평점: 7.373\n   ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)\n\n3. **[Hoppers](https://www.themoviedb.org/movie/1327819)**\n   - 개봉일: 2026-03-04\n   - 장르: 애니메이션, 가족, SF, 코미디, 모험\n   - 줄거리: 사람의 의식을 로봇 동물로 전이하는 과학 기술을 통해 동물과 소통하는 이야기입니다.\n   - 평점: 7.6\n   ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)\n\n4. **[The Super Mario Bros. Movie](https://www.themoviedb.org/movie/502356)**\n   - 개봉일: 2023-04-05\n   - 장르: 가족, 코미디, 모험, 판타지, 애니메이션\n   - 줄거리: 브루클린의 배관

In [6]:
run_agent("첫 번째 영화에 대해 더 알려줘")

User: 첫 번째 영화에 대해 더 알려줘



  -> Calling: get_movie_details({'movie_id': 1523145})


Agent: **[Your Heart Will Be Broken](https://www.themoviedb.org/movie/1523145)**에 대한 정보입니다.

- **개봉일:** 2026년 3월 26일
- **장르:** 로맨스, 드라마
- **러닝타임:** 134분
- **국가:** 러시아 (RU)
- **언어:** 러시아어
- **평점:** 6.9 (총 투표수: 55)

### 줄거리
고등학생 폴리나는 새로운 학교에서 괴롭힘을 당하며, 주범인 바르스와 거래를 하게 됩니다. 바르스는 그녀의 남자친구 역할을 하며 그녀를 보호해야 하고, 폴리나는 그의 모든 지시를 따라야 합니다. 이 게임을 통해 그들은 진정한 감정을 키워가지만, 그녀의 가족과 클래스메이트들은 두 연인을 분리할 이유들이 있습니다.

### 제작 회사
- **All Media A Start Company**
- **START Studio**
- **Sverdlovsk Film Studio**
- **Cinema Foundation of Russia**
- **Yellow, Black & White**
- **Premier Studios**

### 포스터
![Your Heart Will Be Broken 포스터](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)

이 영화에 대해 더 궁금한 점이 있으면 언제든지 말씀해 주세요!


'**[Your Heart Will Be Broken](https://www.themoviedb.org/movie/1523145)**에 대한 정보입니다.\n\n- **개봉일:** 2026년 3월 26일\n- **장르:** 로맨스, 드라마\n- **러닝타임:** 134분\n- **국가:** 러시아 (RU)\n- **언어:** 러시아어\n- **평점:** 6.9 (총 투표수: 55)\n\n### 줄거리\n고등학생 폴리나는 새로운 학교에서 괴롭힘을 당하며, 주범인 바르스와 거래를 하게 됩니다. 바르스는 그녀의 남자친구 역할을 하며 그녀를 보호해야 하고, 폴리나는 그의 모든 지시를 따라야 합니다. 이 게임을 통해 그들은 진정한 감정을 키워가지만, 그녀의 가족과 클래스메이트들은 두 연인을 분리할 이유들이 있습니다.\n\n### 제작 회사\n- **All Media A Start Company**\n- **START Studio**\n- **Sverdlovsk Film Studio**\n- **Cinema Foundation of Russia**\n- **Yellow, Black & White**\n- **Premier Studios**\n\n### 포스터\n![Your Heart Will Be Broken 포스터](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)\n\n이 영화에 대해 더 궁금한 점이 있으면 언제든지 말씀해 주세요!'

In [7]:
run_agent("비슷한 영화 추천해 줄래?")

User: 비슷한 영화 추천해 줄래?



  -> Calling: get_similar_movies({'movie_id': 1523145})


Agent: **[Your Heart Will Be Broken](https://www.themoviedb.org/movie/1523145)**와 비슷한 영화 몇 편을 추천해 드립니다.

1. **[A Season of Giants](https://www.themoviedb.org/movie/278756)**
   - 개봉일: 1990-11-04
   - 장르: 드라마, 역사
   - 줄거리: 16세기 전환기에 미켈란젤로, 라파엘, 레오나르도 다 빈치가 각자의 걸작을 창조하는 이야기를 다룹니다.
   - 평점: 6.5
   ![포스터](https://image.tmdb.org/t/p/w780/1N7hZb4Gmnen9h7GNChD299kcUV.jpg)

2. **[North Dallas Forty](https://www.themoviedb.org/movie/29786)**
   - 개봉일: 1979-08-03
   - 장르: 코미디, 드라마
   - 줄거리: 프로 미식축구 선수의 생활을 반영한 반허구적 이야기입니다.
   - 평점: 6.648
   ![포스터](https://image.tmdb.org/t/p/w780/5e9IeZPFmApZZ37ixq9QhYZXURq.jpg)

3. **[Monkey Shines](https://www.themoviedb.org/movie/29787)**
   - 개봉일: 1988-07-29
   - 장르: 공포, SF, 드라마
   - 줄거리: 사지마비를 가진 남자가 훈련된 원숭이의 도움을 받지만, 원숭이가 그의 주인에게 공격성을 드러내게 되는 이야기입니다.
   - 평점: 6.3
   ![포스터](https://image.tmdb.org/t/p/w780/zbxrAWN2Ss5RXK1uDvMW82xFh0n.jpg)

4. **[Thalapathi](https://www.themoviedb.org/movie/29811)**
   - 개봉일: 1991-11-05
   - 장르: 액션, 드라마
   - 줄거리: 슬럼에서 자란 남성이 가

'**[Your Heart Will Be Broken](https://www.themoviedb.org/movie/1523145)**와 비슷한 영화 몇 편을 추천해 드립니다.\n\n1. **[A Season of Giants](https://www.themoviedb.org/movie/278756)**\n   - 개봉일: 1990-11-04\n   - 장르: 드라마, 역사\n   - 줄거리: 16세기 전환기에 미켈란젤로, 라파엘, 레오나르도 다 빈치가 각자의 걸작을 창조하는 이야기를 다룹니다.\n   - 평점: 6.5\n   ![포스터](https://image.tmdb.org/t/p/w780/1N7hZb4Gmnen9h7GNChD299kcUV.jpg)\n\n2. **[North Dallas Forty](https://www.themoviedb.org/movie/29786)**\n   - 개봉일: 1979-08-03\n   - 장르: 코미디, 드라마\n   - 줄거리: 프로 미식축구 선수의 생활을 반영한 반허구적 이야기입니다.\n   - 평점: 6.648\n   ![포스터](https://image.tmdb.org/t/p/w780/5e9IeZPFmApZZ37ixq9QhYZXURq.jpg)\n\n3. **[Monkey Shines](https://www.themoviedb.org/movie/29787)**\n   - 개봉일: 1988-07-29\n   - 장르: 공포, SF, 드라마\n   - 줄거리: 사지마비를 가진 남자가 훈련된 원숭이의 도움을 받지만, 원숭이가 그의 주인에게 공격성을 드러내게 되는 이야기입니다.\n   - 평점: 6.3\n   ![포스터](https://image.tmdb.org/t/p/w780/zbxrAWN2Ss5RXK1uDvMW82xFh0n.jpg)\n\n4. **[Thalapathi](https://www.themoviedb.org/movie/29811)**\n   - 개봉일: 1991-11-05\n   - 장르: 액션, 드라마\n   